# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [25]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

import numpy as np
np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [26]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2026, 1, 1)
END_DATE   = datetime.date(2026, 3, 20)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 15)

COPY_WINDOW_SECONDS = 5 * 60  # 5 minutes

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
ENRICHED_TRADES_DIR  = Path("../../data/trades_polygon_enriched")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [27]:
# Load only the market files whose month falls within [START_DATE, END_DATE].
import datetime as _dt
def _file_in_range(p, start, end):
    """Return True if YYYY-MM.parquet filename falls within the date range."""
    try:
        y, m = (int(x) for x in p.stem.split("-"))
        file_date = _dt.date(y, m, 1)
        return start.replace(day=1) <= file_date <= end.replace(day=1)
    except Exception:
        return False

market_files = [
    p for p in sorted(MARKETS_DIR.glob("*.parquet"))
    if _file_in_range(p, START_DATE, END_DATE)
]
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 3 market file(s)
Unique markets (all):  303,101
Unique markets (filtered 2026-01-01 → 2026-03-20): 293,807


,end_date_iso,condition_id,market_json
0,2026-01-01T00:00:00Z,0x006ce0742c3891c396aead079e563c5d58c4eae2f9b05d910ed45110980290ad,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-10-20T21:13:17Z"",""minimum_order_size"":5,""minimum_tick_size"":0.001,""condition_id"":""0x006ce0742c3891c396aead079e563c5d58c4eae2f9b05d910ed45110980290ad"",""question_id"":""0x7cf043f4f8d4d781bdcedc859c3c8360936e8d2846b6c136bf722e192da06825"",""question"":""Will Rasmr win the Synthetix trading competition?"",""description"":""This market will resolve to the listed participant with the highest PnL at the end of the Synthetix trading competition starting on October 20, 2025.\n\nThe resolution source will be the Synthetix Leaderboard on https://www.synthetix.io/.\n\nIf two or more participants are tied for the highest PnL at the time of resolution, the market will resolve to the participant whose display name appears first in alphabetical order on the official Synthetix leaderboard.\n\nIf the competition is not completed by December 31, 2025, 11:59 PM ET, or if the official leaderboard or results are unavailable, the market will remain open until reliable data is published by Synthetix. If no such data is available by December 31, 2025, the market will resolve to “Other.”\n\nNote: The official competition includes 100 participants, but this market will resolve solely based on a selected group of KOL participants. Traders who are not eligible KOLs will not be considered. E.g. an eligible KOL will be considered to have won if they are the top trader out of eligible KOLs regardless of if one of the non KOL participants finishes ahead of them. \n\nAdditional KOLs may be added as options later to reflect the full set of eligible KOLs."",""market_slug"":""will-rasmr-win-the-synthetix-trading-competition-933"",""end_date_iso"":""2026-01-01T00:00:00Z"",""game_start_time"":null,""seconds_delay"":0,""fpmm"":"""",""maker_base_fee"":0,""taker_base_fee"":0,""notifications_enabled"":true,""neg_risk"":true,""neg_risk_market_id"":""0x7cf043f4f8d4d781bdcedc859c3c8360936e8d2846b6c136bf722e192da06800"",""neg_risk_request_id"":""0xda26edfbf8e42092e0693c8f4cedd5319c371ff9bd8650f2d7d7b00b19ce6fa2"",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/who-will-win-the-synthetix-trading-competition-TuVGkGEsLmbc.png"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/who-will-win-the-synthetix-trading-competition-TuVGkGEsLmbc.png"",""rewards"":{""rates"":null,""min_size"":50,""max_spread"":3.5},""is_50_50_outcome"":false,""tokens"":[{""token_id"":""67487832607016995738972990163663499867372390570812445168484453192396152205498"",""outcome"":""Yes"",""price"":0,""winner"":false},{""token_id"":""99393257971182825969115164966386446587421508634202783327226489236020763806664"",""outcome"":""No"",""price"":1,""winner"":true}],""tags"":[""Crypto""]}"
1,2026-01-01T00:00:00Z,0x008add40355561724afbce56f68f1aa4ca4d83d8bdba3ed397f03a2743b47dd9,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-12-31T14:01:53Z"",""minimum_order_size"":5,""minimum_tick_size"":0.01,""condition_id"":""0x008add40355561724afbce56f68f1aa4ca4d83d8bdba3ed397f03a2743b47dd9"",""question_id"":""0x9c4aa152472d5b3332f1760add02bb13dc6804e1fe4021e35d8a3ecd8d1a4e29"",""question"":""Bitcoin Up or Down - January 1, 9:00AM-9:15AM ET"",""description"":""This market will resolve to \""Up\"" if the Bitcoin price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to \""Down\"".\nThe resolution source for this market is information from Chainlink, specifically the BTC/USD data stream available at https://data.chain.link/streams/btc-usd.\nPlease note that this market is about the price according to Chainlink data stream BTC/

In [28]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    skipped_markets = 0

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            skipped_markets += 1
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        has_winner = False

        for tok in m.get("tokens", []):
            w = bool(tok.get("winner", None))
            if(w):
                has_winner = True
                break

        if not has_winner:
            skipped_markets += 1
            continue


        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", None))

            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })
    
    print(f"Skipped markets due to JSON parsing or missing winner flag: {skipped_markets}")

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Skipped markets due to JSON parsing or missing winner flag: 21146
Token lookup entries: 545,322
Market meta rows:     272,661


## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel.  Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal.  The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

In [29]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.shard_processor import (
    select_top_wallets_shard,
    enrich_and_group_shard,
)

trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade shard(s)")

sample_raw = pd.read_parquet(trade_files[0])
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)

Found 16 trade shard(s)
Sample shard rows (0.parquet): 3,845,875
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'token_winner', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']
Date range: 2025-12-01 → 2026-03-20


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,token_winner,wallet,side,price,quantity,usdc_amount,question,slug,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0x607ae407a5a4f118d365ec2f60330e1b05eeaa2a0b70f3a1db37d014deb0bff0,1339,79721379,1764547796,2025-12-01,0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b,66798947502955057018675232226813833342419728028280587637631649526290269542515,Yes,False,0xcfb8408de777bd2634e6d684bd3c08b90ba1ec8d,BUY,0.33,139.00,45.87,Will Adam Sandler be nominated for Best Supporting Actor at the 98th Academy Awards?,will-adam-sandler-be-nominated-for-best-supporting-actor-at-the-98th-academy-awards,1,139.00,False,False,"[Awards, Movies, Culture, Oscars]",BUY,Yes,66798947502955057018675232226813833342419728028280587637631649526290269542515,0.33,45.87
1,0x607ae407a5a4f118d365ec2f60330e1b05eeaa2a0b70f3a1db37d014deb0bff0,1339,79721379,1764547796,2025-12-01,0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b,55232573152290628608597793176937541002207221535726193170042948523765565994825,No,True,0xc88694a2c8b0bdf3abc54e6a08a0b78137a9696f,BUY,0.67,139.00,93.13,Will Adam Sandler be nominated for Best Supporting Actor at the 98th Academy Awards?,will-adam-sandler-be-nominated-for-best-supporting-actor-at-the-98th-academy-awards,1,139.00,True,False,"[Awards, Movies, Culture, Oscars]",SELL,Yes,66798947502955057018675232226813833342419728028280587637631649526290269542515,0.33,45.87
2,0xaccb1ae033d686da55aa05062efd4a2025a95bd1d3d65c81a3de227ecadc9be2,1487,79721381,1764547800,2025-12-01,0x00a012e8f00c79862d86ceb1a9cf74d1a57059fbd69634008c95c9d5495b98fd,5840973813263295343327206180777230572868954396024139068969226595774805102694,Yes,False,0xe17a0bfb659a6058bbf39d25d55b4781c2abc6fb,BUY,0.22,36.00,7.92,Will Guillermo del Toro be nominated for Best Director at the 98th Academy Awards?,will-guillermo-del-toro-be-nominated-for-best-director-at-the-98th-academy-awards,1,36.00,False,False,"[Awards, Movies, Culture, Oscars]",BUY,Yes,5840973813263295343327206180777230572868954396024139068969226595774805102694,0.22,7.92


In [30]:
# Preprocess: enrich by copyable quantity if not present

from concurrent.futures import ProcessPoolExecutor

from polymarket_analysis.preprocessing.fill_extender import enrich_shard

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(enrich_shard, f, ENRICHED_TRADES_DIR, COPY_WINDOW_SECONDS): f for f in trade_files}

    total = len(futures)

    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error processing {f}: {e}")
            continue

        if i % 4 == 0 or i == total:
            print(f"{i}/{total} shards processed")

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

Enriched file for 0.parquet already exists, skipping...
Enriched file for 1.parquet already exists, skipping...
Enriched file for 6.parquet already exists, skipping...
Enriched file for 7.parquet already exists, skipping...
Enriched file for 8.parquet already exists, skipping...


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


Enriched file for 9.parquet already exists, skipping...
Enriched file for 2.parquet already exists, skipping...
4/16 shards processed
Enriched file for a.parquet already exists, skipping...
Enriched file for b.parquet already exists, skipping...
Enriched file for c.parquet already exists, skipping...
8/16 shards processed
Enriched file for e.parquet already exists, skipping...
Enriched file for f.parquet already exists, skipping...
12/16 shards processed
Enriched file for 3.parquet already exists, skipping...
Enriched file for 4.parquet already exists, skipping...


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


Enriched file for 5.parquet already exists, skipping...
Enriched file for d.parquet already exists, skipping...
16/16 shards processed


In [31]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

MARKET = '0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b'

df = pd.read_parquet(ENRICHED_TRADES_DIR / "enriched_0.parquet")
df = df[df['condition_id'] == MARKET]
# df[['side', 'price', 'quantity', 'ts', 'token_id', 'tx_hash', 'wallet', 'position']].head(5)

In [32]:
# len(df)
# df[df['tx_hash']=='0xffece5c5cde2b0e1b9375cf30cbb3af09f087967143aff3b4fe5e53c8d1b58c3']

In [33]:
# df['ts'] = pd.to_datetime(df['block_timestamp'], unit='s')
# df['token_id'] = df['token_id'].str[:5]
# df['tx_hash'] = df['tx_hash'].str[:5]
# df[["wallet", 'tx_hash', 'ts', "side", "price", "quantity","token_id", "token_winner", "avail_copy_qty", "avail_copy_total_vol", "avail_copy_count","condition_id"]].head(1)

In [34]:
# ── build token resolution DataFrame ────────────────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard...")
shard_wallet_pnl: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(select_top_wallets_shard, f, token_df, END_TRAIN_TS, top_pct=0.04): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnl or pnl > shard_wallet_pnl[w]:
                shard_wallet_pnl[w] = pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(
                f"  {i}/{len(enriched_trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnl):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnl.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")

Phase 1 — selecting top-4% wallets per shard...


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

Processing shard enriched_0.parquet...
Processing shard enriched_1.parquet...


0.02s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


Processing shard enriched_2.parquet...


...

Processing shard enriched_e.parquet...
  8/16 shards | raw: 32,368,896 | in-range: 30,034,852 | wallet union so far: 18,719
Processing shard enriched_f.parquet...


/Users/vobornij/projects/polymarket/src/polymarket_analysis/preprocessing/shard_processor.py:104: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.loc[:, "copyable_pnl"] = (
/Users/vobornij/projects/polymarket/src/polymarket_analysis/preprocessing/shard_processor.py:104: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.loc[:, "copyable_pnl"] = (


  12/16 shards | raw: 49,004,563 | in-range: 45,606,702 | wallet union so far: 24,439


/Users/vobornij/projects/polymarket/src/polymarket_analysis/preprocessing/shard_processor.py:104: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.loc[:, "copyable_pnl"] = (
/Users/vobornij/projects/polymarket/src/polymarket_analysis/preprocessing/shard_processor.py:104: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.loc[:, "copyable_pnl"] = (
/Users/vobornij/projects/polymarket/src/polymarket_analysis/preprocessing/shard_processor.py:104: SettingWithCopyWarning: 
A value is trying to be set on a co

  16/16 shards | raw: 65,319,633 | in-range: 60,329,636 | wallet union so far: 28,118

Phase 1 done. Candidate wallets (union of top-4% per shard): 28,118


## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [35]:
# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training P&L per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(enrich_and_group_shard, f, token_df, END_TRAIN_TS, top_wallets): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(f"  {i}/{len(enriched_trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
        copyable_pnl     = ("copyable_pnl",     "sum"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)

Phase 2 — grouping and filtering shards...


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 29,285,477
  Unique wallets in grouped:                   28,118


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,copyable_pnl,copyable_qty,avail_copy_total_vol,avail_copy_count,avg_price
0,0xa43e7f0f8c8d765da347395ebe97f9d8aaa34df3e0d079e2e0704c255003013c,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.00,21.20,8.91,3.92,8.91,1,4.99,4.99,8.91,2451.72,89.00,0.44
1,0xb16ce0318e26bccd2b3fa98572a1f3e953941319a65ea2f1338c79d15210413f,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.00,12.29,12.29,5.41,12.29,1,6.88,6.88,12.29,2439.71,87.00,0.44
2,0xe7300a94643440ab691becc1122b0816b8944d99b6f7ed2a2036e6fcbd1f9125,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.00,99.98,50.36,22.16,50.36,1,28.20,28.20,50.36,2343.46,82.00,0.44
3,0x3604ec14512d8ee691ad062768743212fdc70245ce5998cb26da7df8feeca8b2,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.00,49.62,28.43,12.51,28.43,1,15.92,15.92,28.43,2343.46,82.00,0.44
4,0xd89bb82e1dd92941fc3c9cc1636b2f6582bd21508f1195c6d1e76df9f5a54edc,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:19:10+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,True,1.00,784.60,684.62,356.00,684.62,2,328.62,328.62,684.62,17631.12,478.00,0.52


## 4 · Phase 3: apply final PnL cut-off and export

Compute the true cross-shard training P&L per wallet.  Use the **median** of the
per-shard P&L values accumulated in Phase 1 as the export cut-off: wallets whose
total training P&L falls below that median are dropped before writing.

In [36]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",        "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        copyable_pnl_usdc        = ("copyable_pnl",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
    )
    .sort_values("copyable_pnl_usdc", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard pnl values collected in Phase 1 ─────
import statistics
shard_pnl_values = list(shard_wallet_pnl.values())
pnl_cutoff = statistics.median(shard_pnl_values)

final_wallets = set(
    wallet_summary.loc[wallet_summary["copyable_pnl_usdc"] >= pnl_cutoff, "wallet"]
)

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard PnL median (cut-off):        {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing PnL cut-off:           {len(final_wallets):,}")
wallet_summary.head(5)

Unique wallets after Phase 2 union:    28,118
Per-shard PnL median (cut-off):        438.67 USDC
Wallets passing PnL cut-off:           8,965


,wallet,num_markets,num_trades,total_cost_usdc,copyable_pnl_usdc,trade_pnl
0,0xe20a1538293903b746ffe6c4ce2d5c3c0300e469,1171,29636,30259307.20,940507.25,239682.91
1,0x07b8e44b90cc3e91b8d5fe60ea810d2534638e25,182,20476,14395104.80,723949.17,521486.41
2,0x91654fd592ea5339fc0b1b2f2b30bfffa5e75b98,115,5752,10607844.32,600729.28,1386753.83
3,0x876426b52898c295848f56760dd24b55eda2604a,76,10113,3927546.05,588511.05,1554147.98
4,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,319,50996,26667766.54,584540.64,2796901.66


## 5 · Market-level volume summary

In [37]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 136,979


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3a154603dc03c02b57de5,US government shutdown Saturday?,2026-01-31 00:00:00+00:00,3123,223609,124149816.30,128164573.38
1,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Khamenei out as Supreme Leader of Iran by February 28?,2026-02-28 00:00:00+00:00,2507,134884,86891373.30,93791872.92
2,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,"US strikes Iran by February 28, 2026?",2026-01-31 00:00:00+00:00,3002,160057,58121073.07,64022233.52
3,0xa8b744720006da3c08b4dc8a61a5ce930542f550fcf8d27380ae898de636799d,Khamenei out as Supreme Leader of Iran by January 31?,2026-01-31 00:00:00+00:00,2470,104177,39856606.59,41216424.76
4,0xabb86b080e9858dcb3f46954010e49b6f539c20036856c7f999395bfd58d01e6,"US strikes Iran by January 31, 2026?",2026-01-31 00:00:00+00:00,3288,170183,33830761.58,34885699.84
5,0x204d24f3a0f5dd5fca825292bdeab6a97af3978b2caa2b21bb37e610eddfff5d,Seahawks vs. Patriots,2026-02-08 00:00:00+00:00,2549,83345,32003721.46,31944855.61
6,0xb3dbb0f2e1df15cc820e0cde749eb112bf160994ec53a8ac57dceff9dbe5007e,"Will Melania say ""Career"" during AI talk on Friday?",2026-01-16 00:00:00+00:00,233,6766,27133917.35,27163704.74
7,0x05beef793157deb1cc34e2d3159539f461b73c7eeaa3b5348617c10eed5509d0,U.S. anti-cartel ground operation in Mexico by January 31?,2026-01-31 00:00:00+00:00,590,18755,26275059.61,26430366.08
8,0xb8c1bd306a8a4cedfb280e114e655c5092b3f37edccae05cd877d7f21a5774ce,"Russia x Ukraine ceasefire by January 31, 2026?",2026-01-31 00:00:00+00:00,995,51675,18805809.63,18851294.56
9,0x41e47408f8ab39b46a9d9e3c9b15ebd62f1d795eb072ff46df3d376c09eb583e,"US x Iran meeting by February 6, 2026?",2026-02-13 00:00:00+00:00,580,22608,17437983.70,17622926.15


## 6 · Wallet statistics (quantiles)

In [38]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl_usdc", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl_usdc", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl_usdc", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl_usdc", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl_usdc"]    = wallet_stats["copyable_pnl_usdc"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl_usdc,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,28,1.00,1.00,-237180.25,-394605.14,28090
0.01,281,1.00,1.00,-31953.27,-35618.79,27837
0.05,1406,3.00,1.00,-5338.48,-6648.91,26712
0.10,2812,6.00,2.00,-2051.49,-2735.62,25306
0.25,7030,22.00,5.00,-406.31,-635.42,21088
0.50,14059,57.00,14.00,190.40,184.14,14059
0.75,21088,279.00,43.00,602.86,1074.56,7030
0.90,25306,1061.00,155.00,1607.47,3453.54,2812
0.95,26712,2413.40,301.00,3281.45,7597.61,1406


## 7 · Full enriched trade table

In [39]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills
0,0xa43e7f0f8c8d765da347395ebe97f9d8aaa34df3e0d079e2e0704c255003013c,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,21.20,8.91,0.44,True,1.00,3.92,8.91,4.99,4.99,1
1,0xb16ce0318e26bccd2b3fa98572a1f3e953941319a65ea2f1338c79d15210413f,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:54+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,12.29,12.29,0.44,True,1.00,5.41,12.29,6.88,6.88,1
2,0xe7300a94643440ab691becc1122b0816b8944d99b6f7ed2a2036e6fcbd1f9125,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,99.98,50.36,0.44,True,1.00,22.16,50.36,28.20,28.20,1
3,0x3604ec14512d8ee691ad062768743212fdc70245ce5998cb26da7df8feeca8b2,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:16:56+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,49.62,28.43,0.44,True,1.00,12.51,28.43,15.92,15.92,1
4,0xd89bb82e1dd92941fc3c9cc1636b2f6582bd21508f1195c6d1e76df9f5a54edc,0x000269109ef14628e4a4a406acc548487ff1103e,BUY,2026-01-29 13:19:10+00:00,0x925d165d0e799b100fbda20c9417e668901c14458405fb43a1ea8114f40ac3be,Weibo Gaming,784.60,684.62,0.52,True,1.00,356.00,684.62,328.62,328.62,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29285472,0x89109249657c61d7107226ea2fd8803200bc5b83d07a083328f53823d6b984f5,0xfffee90a12347161a770575e5d4f0a1a0da64e6c,SELL,2026-03-13 17:06:41+00:00,0x267dc669f23f6c1ec8cb3973cdd5ecccb250fe941296baaf57b1a629feaa0d2c,FUT Esports,96.39,1.50,0.31,False,0.00,0.47,0.00,0.47,0.46,1
29285473,0xe731874460816433d7c6800ad22fc2fa27d43ba0afc8d782e5bd0beebedfa0ee,0xfffee90a12347161a770575e5d4f0a1a0da64e6c,SELL,2026-03-13 17:06:43+00:00,0x267dc669f23f6c1ec8cb3973cdd5ecccb250fe941296baaf57b1a629feaa0d2c,FUT Esports,96.11,0.28,0.31,False,0.00,0.09,0.00,0.09,0.09,1
29285474,0x0f3090ce0bed6d8477e65b1c6bc7ab788c2e77f8e9ff4e5bbbcb3959ff16e397,0xfffee90a12347161a770575e5d4f0a1a0da64e6c,SELL,2026-03-13 17:06:43+00:00,0x267dc669f23f6c1ec8cb3973cdd5ecccb250fe941296baaf57b1a629feaa0d2c,FUT Esports,95.83,0.28,0.31,False,0.00,0.09,0.00,0.09,0.09,1
29285475,0xf2c2ac524806dd8268e2653c0fd259f9e09c21cf677979c2460f7dd28090352c,0xfffee90a12347161a770575e5d4f0a1a0da64e6c,SELL,2026-03-13 17:06:45+00:00,0x267dc669f23f6c1ec8cb3973cdd5ecccb250fe941296baaf57b1a629feaa0d2c,FUT Esports,95.27,0.28,0.31,False,0.00,0.09,0.00,0.09,0.09,1


## 8 · Export: apply PnL cut-off and write partitioned parquet

Filter grouped trades to ``final_wallets`` (wallets above the median per-shard PnL),
then write one parquet shard per first hex character of ``condition_id`` after ``0x``.

In [40]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 8,965


In [41]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train","copyable_qty", "avail_copy_total_vol", "avail_copy_count",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  5,958,115
  training rows: 3,716,622
  test rows:     2,241,493
Output shards:  16
  0.parquet  (327,815 rows)
  1.parquet  (340,941 rows)
  2.parquet  (389,261 rows)
  3.parquet  (394,643 rows)
  4.parquet  (457,297 rows)
  5.parquet  (352,104 rows)
  6.parquet  (411,815 rows)
  7.parquet  (373,898 rows)
  8.parquet  (345,207 rows)
  9.parquet  (336,945 rows)
  a.parquet  (426,189 rows)
  b.parquet  (370,021 rows)
  c.parquet  (324,871 rows)
  d.parquet  (385,087 rows)
  e.parquet  (363,580 rows)
  f.parquet  (358,441 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [42]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

In [43]:
pd.read_parquet('../../data/polygon_trades_processed/0.parquet').head(5)

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count
0,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x062e5937100103295e98f56a99926df33f05e3b9f1a1929780aef1f4c46d73e2,2026-01-05 18:07:31+00:00,BUY,No,6281.66,6281.66,0.98,6152.59,6281.66,129.07,13.14,True,1.00,0xc103aa8b6f1a369e6b704a2eaac3b938f95daf1bdf679cb63af7841143b82035,11,True,639.66,16.52,13.00
1,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x062e5937100103295e98f56a99926df33f05e3b9f1a1929780aef1f4c46d73e2,2026-01-05 19:18:25+00:00,BUY,No,7585.45,1303.79,0.98,1272.70,1303.79,31.09,0.00,True,1.00,0x6b01ff8c7f4b4e95d6aa2516b3a56dd15e19596deabd4c863d09e4484ff61145,7,True,-0.00,0.00,0.00
2,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x062e5937100103295e98f56a99926df33f05e3b9f1a1929780aef1f4c46d73e2,2026-01-05 22:19:01+00:00,BUY,No,9419.77,1834.32,0.98,1799.14,1834.32,35.18,35.18,True,1.00,0x8d0f6a5e5fecdc5a65fb00513f01aeb69992077fc9aaf079a7ea554d1f1df6bf,8,True,1834.32,662.87,40.00
3,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x062e5937100103295e98f56a99926df33f05e3b9f1a1929780aef1f4c46d73e2,2026-01-05 22:21:17+00:00,BUY,No,10269.13,849.36,0.98,834.07,849.36,15.29,0.00,True,1.00,0x7cffad1ee424cc997526fcb69cc35b44690bb9213feac6df1db9f2efb8a45dec,2,True,-0.00,0.00,0.00
4,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x062e5937100103295e98f56a99926df33f05e3b9f1a1929780aef1f4c46d73e2,2026-01-05 22:24:41+00:00,BUY,No,10380.24,111.11,0.98,109.11,111.11,2.00,0.00,True,1.00,0xe10b9e5832bf071da9f60146e779476dd52d77ee858f5ee5bd70546bed549995,1,True,-0.00,0.00,0.00


### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.

In [44]:
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")

INFO  CTF Exchange rows not present in current filtered dataset
PASS  CTF Exchange not in top_wallets: top_wallets has 8,965 entries
